# Recover Traffic Sign Recognition
This notebook restores saved weights and bundled DINOv2 code. It does not download the 1.48 GB source dataset or run training. A T4 is optional; CPU inference is supported.

1. Run the download cell, or upload your complete saved project ZIP when prompted.
2. Run the launch cell.
3. Open the temporary Gradio link.

Camera snapshots require browser permission. Model scores are uncalibrated decimal outputs, not guaranteed correctness.

In [ ]:
# Recover the published bundle or upload your saved project ZIP
import pathlib,urllib.request,hashlib,zipfile,io
ROOT=pathlib.Path('/content/traffic_sign_recovery');ROOT.mkdir(exist_ok=True)
base='https://github.com/futurefokus/traffic-sign-recognition-kazakhstan/releases/download/v1.0/'
try:
    raw=urllib.request.urlopen(base+'Traffic_Sign_Project.zip',timeout=180).read()
    expected=urllib.request.urlopen(base+'Traffic_Sign_Project.zip.sha256',timeout=30).read().decode().split()[0]
    assert hashlib.sha256(raw).hexdigest()==expected,'Checksum mismatch'
except Exception as error:
    print('Release download unavailable. Upload the complete saved project ZIP.',type(error).__name__)
    from google.colab import files
    uploaded=files.upload()
    matches=[name for name in uploaded if name.lower().endswith('.zip')]
    assert len(matches)==1,'Upload exactly one project ZIP'
    raw=uploaded[matches[0]]
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for info in z.infolist():
        if info.is_dir():continue
        target=(ROOT/info.filename).resolve()
        assert ROOT in target.parents,'Unsafe archive path'
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(info))
print('Restored project to',ROOT)


In [ ]:
# Install runtime dependencies and start from saved weights only
import sys,subprocess,runpy
subprocess.check_call([sys.executable,'-m','pip','-q','install','-r',str(ROOT/'requirements.txt')])
restored=runpy.run_path(str(ROOT/'app.py'),run_name='saved_weight_app')
demo=restored['demo']
demo.launch(share=True,debug=False,prevent_thread_lock=True,js=restored['ENGLISH_JS'],css='footer {display:none !important}')
restored['force_english'](demo)
